In [1]:
# importing necessary python modules for pyspark execution
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,\
                            trim, upper, avg, round,\
                            sum as spark_sum, to_date, when, count,\
                            current_timestamp, regexp_replace, regexp,\
                            broadcast, round as spark_round, max as spark_max, mode,\
                            lit, desc, collect_list, min as spark_min, median as spark_median
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType, FloatType

In [ ]:
##################################################################################################################
###################################### SparkSession Creation function ############################################
########################## This helps to create sparksession #####################################################
##################################################################################################################

In [2]:
# Create helper static fucntion for creating spark session
@staticmethod
def create_spark_session():
    spark = SparkSession.builder\
                         .appName('Mini_Assignment2_Session')\
                         .getOrCreate()
    return spark

In [ ]:
##################################################################################################################
###################################### Metadata helper class #####################################################
########################## This helps to get the metadta of the dataframe ########################################
##################################################################################################################

In [3]:
# Lets create a helper class which provide metadata information of the dataframe
class extractMetadata:
    """
    This class will get dataframe as input and extract its metadata information
    """
    def __init__(self, df):
        """
        Initialize with dataframe --> used in getting the metadata
        Initialize with dataframe's row count and columns --> which provides the dataframe shape
        """
        self.df = df
        self.row_count = df.count()
        self.columns = df.columns
    # Below function will get the dataframe shape and return that as tuple (row_count, column_count)
    def _get_dataframe_shape(self):
        return (self.row_count, len(self.columns))
    # Below function will get the column names and store that in a python list
    def _get_column_names(self):
        return self.df.columns
    # Below function will get the column data types and store that in a python list
    def _get_column_types(self):
        return [datatype[1] for datatype in self.df.dtypes]
    # Below function will get the missing values count for each column and store that in a python list
    def _get_missing_column_counts(self):
        return [self.df.filter(col(column).isNull()).count() for column in self.columns]
    # Below function will get the missing values proportion for each column and store that in a python list
    def _get_missing_column_proportion(self):
        return [self.df.filter(col(column).isNull()).count()/self.row_count for column in self.columns]
    # Below function will get the distinct count of each column and store that in a python list
    def _get_unique_column_counts(self):
        return [self.df.select(col(column)).groupBy(col(column)).count().select(column).count() for column in self.columns]
    # Below function will get index column possibility and store that in a python list
    def _index_column_possiblity(self):
        return ["All Unique" if self.df.select(col(column)).groupBy(col(column)).count().select(column).count() == self.row_count else "Non Unique Column" for column in self.columns]
    # Below function will get the categorical columns top 3 mode value and store that in a python list
    def _get_categorical_mode_values(self):
        categorical_list = []
        for field in self.df.schema.fields:
            if field.dataType not in (DoubleType(),IntegerType()):
                top_3_rows = (
                self.df.groupBy(col(field.name))
                .count()
                .orderBy(desc('count'))
                .head(3)
                )
                top_3_values = [f'{row[0]}-{row[1]}' for row in top_3_rows]
                categorical_list.append(top_3_values)
            else:
                categorical_list.append(['Not a categorical column'])
        return categorical_list
    # Below function will get the numerical columns min, max, median value and store that in a python list
    def _get_numerical_distribution_values(self):
        numerical_list = []
        for field in self.df.schema.fields:
            if field.dataType in (DoubleType(),IntegerType()):
                dist_dict = ([
                            f'min - {self.df.select(spark_min(field.name)).collect()[0][0]}, median - {self.df.select(spark_median(field.name)).collect()[0][0]}, max - {self.df.select(spark_max(field.name)).collect()[0][0]}'
                            ])
                numerical_list.append(dist_dict)
            else:
                numerical_list.append(([f'{field.name}: Not a numerical column']))
        single_column_data = [(item,) for item in numerical_list]
        return single_column_data
    def extract_metadata(self, spark):
        metadata = list(zip(self._get_column_names(),
                            self._get_column_types(),
                            self._get_missing_column_counts(),
                            self._get_missing_column_proportion(),
                            self._get_unique_column_counts(),
                            self._index_column_possiblity(),
                            self._get_categorical_mode_values(),
                            self._get_numerical_distribution_values()
                           )
                       )
        columns = ["column_name",
                   "data_type",
                   "missing_column_count",
                   "missing_column_proportion(%)",
                   "unique_column_count",
                   "index_column_possibility",
                   "top_3_values",
                   "numerical_column_distribution"
                  ]
        return spark.createDataFrame(metadata,
                                    columns)

In [ ]:
##################################################################################################################
###################################### sparkFunctions function ###################################################
########################## This helps to read csv file, tsv file #################################################
########################## This helps to changing the datatype(standardizing the datatypes) ######################
########################## This helps to removing the spaces from the columns ####################################
########################## This helps to standardize null values #################################################
##################################################################################################################

In [4]:
class sparkFunctions:
    """
    This class 
        read CSV file, 
        standardize the columns,
        remove duplicates,
        remove unwanted columns,
        remove unwanted rows
    """
    def __init__(self, spark):
        self.spark = spark
    def read_csv_file(self, input_file, schema=None):
        """
        This function will read the data from comma seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def read_tsv_file(self, input_file, schema=None):
        """
        This function will read the data from tab seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, sep='\t', inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def remove_specific_strings_from_numeric(self, df, column_dict):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column_key,column_value in column_dict.items():
            standardized_df = standardized_df.withColumn(column_key,regexp_replace(col(column_key), column_value, ""))
        return standardized_df
    def remove_any_string_from_numeric(self, df, col_list):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             when(col(column).rlike("^[^0-9]+$"), None).otherwise(col(column))
                                           )
        return standardized_df
    def remove_spaces_from_string(self, df, col_list):
        """
        This function will remove the leading and trailing spaces from string columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             trim(col(column))
                                           )
        return standardized_df
    def standardize_null_values(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        standardized_df = standardized_df.fillna(column_dict)
        return standardized_df
    def standardize_datatypes(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        for column_name, column_type in column_dict.items():
            standardized_df = standardized_df.withColumn(column_name, col(column_name).cast(column_type))
        return standardized_df

In [5]:
# Create spark session
# Lets call the helper function to create the spark session
spark=create_spark_session()

In [6]:
# Let's read the csv file using the sparkFunction helper class
df = sparkFunctions(spark).read_csv_file('Data/Flight Dataset - CSV(in).csv')

In [7]:
# Read the metadata using metadata helper class and keep this ready in the dataframe for future analysis
df_metadata = extractMetadata(df).extract_metadata(spark)

In [8]:
df_metadata.show(truncate=False)

+-----------+---------+--------------------+----------------------------+-------------------+------------------------+------------------------------------------------+--------------------------------------------------+
|column_name|data_type|missing_column_count|missing_column_proportion(%)|unique_column_count|index_column_possibility|top_3_values                                    |numerical_column_distribution                     |
+-----------+---------+--------------------+----------------------------+-------------------+------------------------+------------------------------------------------+--------------------------------------------------+
|FL_DATE    |string   |0                   |0.0                         |59                 |Non Unique Column       |[1/6/2006-19553, 1/5/2006-19534, 1/9/2006-19483]|{[FL_DATE: Not a numerical column]}               |
|DEP_DELAY  |int      |0                   |0.0                         |684                |Non Unique Column       |[Not a

In [ ]:
# With the given data set, solve the following tasks using PySpark.  
# Task 1: Create a function that gives back how many flights arrived earlier than expected. 
# Task 2: Create a function that determines the typical departure time for flights over 2000 miles. 
# Task 3: Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes. 
# Task 4: Create a function that gives the average airtime for flights that left earlier than 9:00 am. 
# Task 5: Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure. 

In [ ]:
################################################################
################# Mini Assignment - 2 Started ##################
################################################################

In [19]:
# Task 1: Create a function that gives back how many flights arrived earlier than expected. 
def task1(df):
    result = df.filter(col('ARR_DELAY')<0).count()
    return result

# Task 2: Create a function that determines the typical departure time for flights over 2000 miles. 
def task2(df):
    result = df.filter(col('DISTANCE')>2000).agg(avg(col('DEP_TIME')).alias('average')).collect()[0][0]
    return result

# Task 3: Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes. 
def task3(df):
    ### Assumption: timings are in minutes ###
    result = df.filter(col('ARR_DELAY')>60).count()
    return result/extractMetadata(df)._get_dataframe_shape()[0]

# Task 4: Create a function that gives the average airtime for flights that left earlier than 9:00 am. 
def task4(df):
    result = df.filter(col('DEP_TIME')<9).agg(avg('AIR_TIME').alias('average')).collect()[0][0]
    return result

# Task 5: Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure. 
def task5(df):
    result = df.filter(col('DEP_DELAY')<=0).select('ARR_DELAY').agg(spark_max(col('ARR_DELAY')).alias('maximum_arrival_delay')).collect()[0][0]
    return result

In [20]:
# Calling helper functions
print(f"-"*45)
print(f"Mini Assignment 2 - Task 1")
print(f"-"*45)
print(f"Number of flights arrived earlier than expected: {task1(df)}")
print(f"-"*45)
print(f"Mini Assignment 2 - Task 2")
print(f"-"*45)
print(f"Average departure time for flights over 2000 miles: {task2(df)}")
print(f"-"*45)
print(f"Mini Assignment 2 - Task 2")
print(f"-"*45)
print(f"Proportion of flights that have arrival delays longer than 60 minutes: {task3(df)}")
print(f"-"*45)
print(f"Mini Assignment 2 - Task 4")
print(f"-"*45)
print(f"Average airtime for flights that left earlier than 9:00 am: {task4(df)}")
print(f"-"*45)
print(f"Mini Assignment 2 - Task 5")
print(f"-"*45)
print(f"Maximum arrival delay for flights that did not experience a delay upon departure: {task5(df)}")

---------------------------------------------
Mini Assignment 2 - Task 1
---------------------------------------------
Number of flights arrived earlier than expected: 534655
---------------------------------------------
Mini Assignment 2 - Task 2
---------------------------------------------
Average departure time for flights over 2000 miles: 13.973233947624635
---------------------------------------------
Mini Assignment 2 - Task 2
---------------------------------------------
Proportion of flights that have arrival delays longer than 60 minutes: 0.053066
---------------------------------------------
Mini Assignment 2 - Task 4
---------------------------------------------
Average airtime for flights that left earlier than 9:00 am: 111.36120276990287
---------------------------------------------
Mini Assignment 2 - Task 5
---------------------------------------------
Maximum arrival delay for flights that did not experience a delay upon departure: 701
